# 08N: Object registry and reusable patch preparation

Reuse the fixed model revision and saved masks. Preserve all valid object candidates and prepare selected-view object banks. Neutral correspondence is an explicitly assisted route. Anomaly labels remain in the evaluator.

## Frozen comparison

LocateAnything-3B, SAM 2.1 and DINOv2-small remain fixed. The existing 100-scene development screen supplies 19 predefined schedules: five each with 1, 2 and 3 views, and four with 5 views. Three and five views are the application settings; one and two views are diagnostics. Every prediction uses only its selected views. This bounded stage does not claim results for 10 or 20 views.

| Route | Object regions | Cross-view identity | Interpretation |
| --- | --- | --- | --- |
| `supplied_reference` | Supplied masks | Supplied neutral correspondence | Scoring diagnostic |
| `automatic_reference` | Saved LocateAnything and SAM outputs | Supplied neutral correspondence | Region diagnostic; assisted identity |
| `automatic_predicted` | Saved LocateAnything and SAM outputs | Existing automatic association | Automatic application route |

Anomaly labels are available only to the evaluator, after predictions are sealed. All valid candidates remain represented. An unmatched observation stays visible in the output; its object-bank score is unavailable because exclusion of another view of itself cannot be guaranteed. Missing observations are never replaced with normal scores. One object cannot contain two detections from the same view.

**Patch comparison.** For each target foreground patch, compute the minimum cosine distance to a peer object's patches across that peer's selected views. Each peer contributes one distance. Take the lower median across peers, then average the largest `max(1, floor(0.02 × target patch count))` patch distances. Exclude the target's complete identity from all peer banks. At least two usable peers are required.

Compare this object bank with regenerated within-view scoring and cached forward scoring. `view_mean_control` first averages nearest-patch distances across each peer's usable views. It diagnoses the extra matching opportunities in a larger bank; it is not a guarantee of invariance to bank size.

**One physical-object ranking.** Convert available observation scores into average within-view percentile ranks when at least three candidates are scored. Compare the mean of all valid ranks with the mean of the strongest two (or the single available rank). Retain the existing winner-consensus method as a comparison. The scorer's raw-score means are diagnostics; final rankings use the declared rank normalization.

**Decision gate.** Choose between the two bank fusions using the supplied-reference primary three-view development result: the paired asset-bootstrap 95% lower bound for top-1 improvement must be positive, AP and Recall@2 must not regress, and the mean top-1 improvement across the five three-view schedules must be positive. Prefer mean fusion on a tie. If the gate fails, stop the scoring extension and retain the stronger existing method. Results remain development evidence; 60% is a goal, not a promised outcome.

Report tie-aware top-1, Recall@2, reciprocal rank and average precision, with asset-grouped uncertainty and explicit candidate, association and ranking losses. A scene contributes one result per setting; its views are not independent successes.

## Runtime and configuration

In [ ]:
from pathlib import Path
import json
import os
import sys

def is_source(path):
    return (path / "scripts/run_object_bank.py").is_file() and (path / "src/vpi_object_bank").is_dir()

def locate_source():
    explicit = os.environ.get("VPI_OBJECT_BANK_ROOT")
    if explicit:
        root = Path(explicit).expanduser().resolve()
        if not is_source(root):
            raise RuntimeError("The configured source root is incomplete")
        return root
    for root in (Path.cwd(), *list(Path.cwd().parents)[:6]):
        if is_source(root):
            return root
    candidates = set()
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for pattern in ("*", "datasets/*/*", "datasets/*/*/versions/*", "notebooks/*/*"):
            for mount in input_root.glob(pattern):
                if not mount.is_dir():
                    continue
                if is_source(mount):
                    candidates.add(mount.resolve())
                children = list(mount.iterdir())
                if len(children) <= 200:
                    candidates.update(child.resolve() for child in children if child.is_dir() and is_source(child))
    if len(candidates) != 1:
        raise RuntimeError("Expected exactly one complete object-bank source resource")
    return candidates.pop()

SOURCE_ROOT = locate_source()
CONFIG_PATH = Path(os.environ.get("VPI_OBJECT_BANK_CONFIG", SOURCE_ROOT / ".local/config.json")).expanduser().resolve()
if not CONFIG_PATH.is_file():
    raise RuntimeError("Provide the verified local or Kaggle configuration JSON")
configuration = json.loads(CONFIG_PATH.read_text())
sys.path.insert(0, str(SOURCE_ROOT / "scripts"))
sys.path.insert(0, str(SOURCE_ROOT / "src"))
from scoring_comparison_runtime import run_bounded
os.environ["PYTHONPATH"] = str(SOURCE_ROOT / "src")
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
LOG_ROOT = SOURCE_ROOT / ".local/logs"
if str(SOURCE_ROOT).startswith("/kaggle/input/"):
    LOG_ROOT = Path("/kaggle/working/object-bank-logs")
LOG_ROOT.mkdir(parents=True, exist_ok=True)

def run_stage(stage):
    run_bounded(
        [sys.executable, str(SOURCE_ROOT / "scripts/run_object_bank.py"), stage, "--config", str(CONFIG_PATH)],
        LOG_ROOT / (stage + ".log"), idle_seconds=900, total_seconds=18000,
    )
print({"event": "notebook_ready", "python": sys.version.split()[0]})

## Input preflight

In [ ]:
run_stage("preflight")

## Reusable patch extraction

In [ ]:
run_stage("extract")

## Object registries and predictions

In [ ]:
run_stage("predict")

## Verify sealed predictions

In [ ]:
run_stage("verify")

## Protocol and reusable cache evidence

In [ ]:
from IPython.display import display
import pandas as pd
from vpi_object_bank.protocol import PROTOCOL

# The protocol has no evaluator labels or machine-specific paths.
display(PROTOCOL)
schedule_rows = [{"schedule": key, "view_count": len(views), "views": ", ".join(views)}
                 for key, views in PROTOCOL["view_sets"].items()]
display(pd.DataFrame(schedule_rows))
feature_root = Path(configuration["feature_root"])
feature_lock = json.loads((feature_root / "feature_lock.json").read_text())
binding = json.loads((feature_root / "run_binding.json").read_text())
progress = json.loads((feature_root / "progress.json").read_text())
metadata = feature_lock.get("metadata", {})
display({"sealed_scene_count": len(metadata.get("scenes", [])),
         "cached_view_count": len(metadata.get("views", [])),
         "completed_scene_region_view_units": metadata.get("completed_units"),
         "last_extraction_progress": {key: progress.get(key) for key in
              ("completed", "total", "extracted", "reused", "elapsed_seconds")},
         "device": binding.get("device"), "runtime": binding.get("runtime"),
         "model_file_sha256": binding.get("model_files")})
print("The extraction cell reports zero extracted units when an identical sealed cache is reused.")